# 03 — Chấm và so sánh A (8B) với B (14B)

Chạy sau khi đã xong notebook 01 và 02.

1. **Giám khảo khác họ** (mặc định `microsoft/phi-4`) giải mù mọi câu do các hệ soạn, và giải cả câu test thật để biết giám khảo tự sai bao nhiêu.
2. **Báo cáo**: tỉ lệ câu dùng được, độ chính xác giải, so sánh cặp có khoảng tin cậy (bootstrap ghép cặp, McNemar), chi phí train/suy luận.

**Tỉ lệ câu dùng được (ước lượng)** = đúng khuôn **và** mọi công thức dựng được bằng KaTeX **và** giám khảo chọn đúng đáp án model ghi **và** không chép gần nguyên văn câu train (Jaccard 5-gram ≥ 0,8).
Đây là cận trên: giám khảo có thể "khớp" với một đáp án sai nếu cả hai cùng sai. Muốn con số cho paper cần giáo viên chấm một mẫu.

In [ ]:
#@title 1. Kiểm tra GPU
import subprocess
info = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                       '--format=csv,noheader,nounits'], capture_output=True, text=True).stdout.strip()
print(info)
name, mem = [x.strip() for x in info.split(',')]
assert 'A100' in name and int(mem) >= 79000, (
    f'Cấu hình batch trong notebook tính cho A100 80GB, runtime hiện tại: {name} {mem} MiB')

In [ ]:
#@title 2. Cấu hình
EXPERIMENTS = 'A=expA_qwen3_8b_seed42,B=expB_qwen3_14b_seed42'  #@param {type:'string'}
JUDGE_MODEL = 'microsoft/phi-4'  #@param {type:'string'}
JUDGE_VOTES = 1  #@param {type:'integer'}
REPO = 'https://github.com/trantrien1/AQG.git'  #@param {type:'string'}
BRANCH = 'lora-finetune'  #@param {type:'string'}
DRIVE_ROOT = '/content/drive/MyDrive/AQG_ft'  #@param {type:'string'}

DATA_DIR = f'{DRIVE_ROOT}/data'
EXPS = [tuple(x.split('=', 1)) for x in EXPERIMENTS.split(',')]
TAG = '_'.join(n for n, _ in EXPS)
JUDGE_DIR = f'{DRIVE_ROOT}/judge_{TAG}_' + JUDGE_MODEL.split('/')[-1] + f'_v{JUDGE_VOTES}'
REPORT_DIR = f'{DRIVE_ROOT}/report_{TAG}'

In [ ]:
#@title 3. Drive, token Hugging Face, hàm chạy lệnh
import os, sys, json, time, shutil, subprocess
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_ROOT, exist_ok=True)

try:  # Colab: 🔑 Secrets -> thêm HF_TOKEN (cần quyền đọc dataset riêng tư)
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception as exc:
    print('Chưa đọc được HF_TOKEN từ Colab Secrets:', exc)

REPO_DIR = '/content/AQG'
NB_DIR = f'{REPO_DIR}/API/notebooks'
ENV = dict(os.environ, PYTHONPATH=NB_DIR, TOKENIZERS_PARALLELISM='false',
           PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')

def sh(cmd, log=None):
    """Chạy lệnh, in đầu ra ngay khi có; lỗi thì dừng notebook."""
    print('$', cmd, flush=True)
    fh = open(log, 'a', encoding='utf-8') if log else None
    p = subprocess.Popen(cmd, shell=True, cwd=NB_DIR if os.path.isdir(NB_DIR) else None,
                         env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, encoding='utf-8', errors='replace')
    for chunk in iter(lambda: p.stdout.read(512), ''):
        sys.stdout.write(chunk)
        if fh:
            fh.write(chunk)
    if fh:
        fh.close()
    if p.wait() != 0:
        raise RuntimeError(f'Lệnh lỗi (mã {p.returncode}): {cmd}')

In [ ]:
#@title 4. Clone repo + cài thư viện (~5 phút)
subprocess.run(['rm', '-rf', REPO_DIR], check=True)
sh(f'git clone -q --depth 1 -b {BRANCH} {REPO} {REPO_DIR} && git -C {REPO_DIR} log --oneline -1')
# vLLM cài trước vì nó ghim phiên bản torch; peft không ghim torch.
sh('pip -q install -U vllm && pip -q install -U "peft>=0.15" pytest')
sh('python -c "import torch, transformers, peft, vllm; '
   'print(\'torch\', torch.__version__, \'| transformers\', transformers.__version__, '
   '\'| peft\', peft.__version__, \'| vllm\', vllm.__version__)"')
sh('python -m pytest -q ../tests/test_mcqft.py')

In [ ]:
#@title Cài KaTeX (kiểm tra công thức trong câu sinh ra)
sh('mkdir -p /content/katex && cd /content/katex && npm install --silent katex@0.16 && ls node_modules | head -3')
KATEX_MODULES = '/content/katex/node_modules'

In [ ]:
#@title 5. Giám khảo giải mù (~5–10 phút)
from huggingface_hub import HfApi
JUDGE_REV = HfApi().model_info(JUDGE_MODEL).sha
preds = ' '.join(f'--preds {n}="{DRIVE_ROOT}/{d}/preds"' for n, d in EXPS)
sh(f'python -m mcqft.judge --model {JUDGE_MODEL} --revision {JUDGE_REV} --data "{DATA_DIR}" '
   f'{preds} --votes {JUDGE_VOTES} --out "{JUDGE_DIR}"', log=f'{JUDGE_DIR}.log')

In [ ]:
#@title 6. Báo cáo
from IPython.display import Markdown, display
exps = ' '.join(f'--exp {n}="{DRIVE_ROOT}/{d}"' for n, d in EXPS)
sh(f'python -m mcqft.report --data "{DATA_DIR}" {exps} --judge "{JUDGE_DIR}" '
   f'--out "{REPORT_DIR}" --katex-modules {KATEX_MODULES}')
display(Markdown(open(f'{REPORT_DIR}/report.md', encoding='utf-8').read()))

In [ ]:
#@title 7. Biểu đồ: chất lượng và chi phí
import matplotlib.pyplot as plt
rep = json.load(open(f'{REPORT_DIR}/report.json', encoding='utf-8'))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
names = list(rep['gen'])
vals = [100 * rep['gen'][n]['usable_rate'] for n in names]
errs = [[100 * (rep['gen'][n]['usable_rate'] - rep['gen'][n]['usable_ci'][0]) for n in names],
        [100 * (rep['gen'][n]['usable_ci'][1] - rep['gen'][n]['usable_rate']) for n in names]]
axes[0].bar(names, vals, yerr=errs, capsize=4); axes[0].set_title('Câu sinh dùng được (%)')
names_s = list(rep['solve'])
vals_s = [100 * rep['solve'][n]['accuracy'] for n in names_s]
errs_s = [[100 * (rep['solve'][n]['accuracy'] - rep['solve'][n]['ci'][0]) for n in names_s],
          [100 * (rep['solve'][n]['ci'][1] - rep['solve'][n]['accuracy']) for n in names_s]]
axes[1].bar(names_s, vals_s, yerr=errs_s, capsize=4); axes[1].set_title('Giải đúng câu test (%)')
exp_names = list(rep['info'])
axes[2].bar([f'{e}\ntrain (phút)' for e in exp_names],
            [(rep['info'][e]['train']['train_seconds'] or 0) / 60 for e in exp_names])
tps = [rep['info'][e]['infer'].get('lora.gen', {}).get('output_tokens_per_second') or 0 for e in exp_names]
ax2 = axes[2].twinx(); ax2.plot([f'{e}\ntrain (phút)' for e in exp_names], tps, 'ro-')
ax2.set_ylabel('token/s khi sinh (LoRA)', color='r'); axes[2].set_title('Chi phí')
for ax in axes[:2]:
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.savefig(f'{REPORT_DIR}/summary.png', dpi=150); plt.show()

## Đọc kết quả

- **LoRA có giúp không?** Xem các cặp `X.base → X.lora` và `X.base_fs3 → X.lora`. Nếu LoRA không hơn few-shot thì chưa cần fine-tune.
- **14B có đáng không?** Xem cặp `A.lora → B.lora`. Với ~166 câu test, chênh lệch nhỏ hơn khoảng 5–7 điểm thường nằm trong nhiễu.
  Nếu khoảng tin cậy chứa 0 (hoặc chênh < 3 điểm) mà 14B train lâu hơn gần gấp đôi và sinh chậm hơn, chọn 8B.
- **Theo mức độ**: 14B thường chỉ hơn rõ ở *Vận dụng cao*; nếu ứng dụng cần nhiều câu khó, cân nhắc riêng phần đó.
- Muốn kết luận chắc hơn: chạy lại 01 và 02 với `SEED` khác (vd. 43, 44) rồi chạy notebook này cho từng cặp seed (vd. `A=expA_qwen3_8b_seed43,B=expB_qwen3_14b_seed43`). Kết luận chỉ đáng tin khi chiều chênh lệch giống nhau ở mọi seed.